# Document Loaders and Text Splitters

In-depth guide to LangChain document loaders and text splitters. Demonstrates how to load documents, parse metadata, configure chunk sizes and overlaps, and compare character-level vs token-level chunking.

## Workflow Architecture

<div align="center">
  <img src="workflow_loaders_and_splitters.png" alt="Document Loaders and Text Splitters Architecture Diagram" width="580" />
</div>

<details>
<summary><b>Click to expand Colorful Mermaid Source Code</b></summary>

```mermaid
flowchart TD
    subgraph Loaders["Document Loaders Layer"]
        PDF["PyPDFLoader / PDFPlumber"]:::loaderNode
        TXT["TextLoader / DirectoryLoader"]:::loaderNode
        CSV["CSVLoader / UnstructuredExcel"]:::loaderNode
    end
    subgraph DocObj["LangChain Document Abstraction"]
        DocObject["Document Objects<br/>page_content + metadata"]:::docNode
    end
    subgraph Splitters["Transformation and Splitting Layer"]
        RecChar["RecursiveCharacterTextSplitter"]:::splitNode
        CharSplit["CharacterTextSplitter"]:::splitNode
        TokenSplit["TokenTextSplitter (tiktoken)"]:::splitNode
    end
    subgraph Output["Search-Ready Chunk Batches"]
        Chunks["Processed Chunks with Inherited Metadata"]:::outNode
    end
    PDF --> DocObject
    TXT --> DocObject
    CSV --> DocObject
    DocObject --> RecChar
    DocObject --> CharSplit
    DocObject --> TokenSplit
    RecChar --> Chunks
    CharSplit --> Chunks
    TokenSplit --> Chunks
    classDef loaderNode fill:#E8F5E9,stroke:#2E7D32,stroke-width:2px,color:#1B5E20;
    classDef docNode fill:#FFF8E1,stroke:#FFA000,stroke-width:2px,color:#E65100;
    classDef splitNode fill:#E0F7FA,stroke:#00838F,stroke-width:2px,color:#004D40;
    classDef outNode fill:#EDE7F6,stroke:#5E35B1,stroke-width:2px,color:#311B92;
```
</details>

### Key Chunking Principles
- **Unified Document Abstraction**: Standardizes raw text and document metadata (source, page, file properties).
- **Configurable Overlap**: Prevents boundary data loss across contiguous chunks.
- **Token-Aware Splitting**: Ensures chunks stay strictly within LLM token window boundaries.


In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_loader = PyPDFLoader("../data/pdf/kafka.pdf")

pdf_documents = pdf_loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=300,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(pdf_documents)

def clean_text(text):
    # Remove extra whitespace and newlines
    cleaned_text = ' '.join(text.strip().split())
    return cleaned_text

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: Length: {len(chunk.page_content)}")
    print(f"Content: {clean_text(chunk.page_content)}...")  # Print the first 100 characters
    print("\n")

C:\Users\itsar\AppData\Local\Temp\ipykernel_51352\2659118441.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\RAG\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunk 0: Length: 980
Content: These are the most relevant contents that I collected through some books and courses. I organized and make some edits to help deliver the concepts about Apache Kafka in the most comprehensible way. Anyone who wants to learn Apache Kafka can reference these notes without going through too many resources on the Internet. Below might be the only material you need to grasp quite a basic understanding of Kafka as well as how to configure your applications to use Kafka properly in production. If you want to get a deeper understanding of Kafka or how to use Kafka Stream for big data processing. I highly recommend you check out the material including some books and courses that I linked in the reference section. If you want a better reading exeperience, visit: https://anhthi.netlify.app/docs/architecture/message_queue/kafka Table of contents: Kafka notes Kafka introduction The data problem Why use Kafka? Why is Kafka fast? Compared to other message queue systems K

In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="../data/chroma_db",
    collection_name="kafka_collection"
)

C:\Users\itsar\AppData\Local\Temp\ipykernel_51352\1143981272.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13296.81it/s]

In [3]:
query = "WhatsApp Chatbot no.?"

similar_docs = vector_db.similarity_search_with_score(query, k=3)
for i, (doc, score) in enumerate(similar_docs):
    print(f"Document {i}:")
    print(f"Score: {score}")
    print(f"Content: {clean_text(doc.page_content)}")  
    print("\n")

Document 0:
Score: 1.6317880153656006
Content: message the leader has When a leader crashes, one of follower replica will be promoted to become the leader A Follower replica that catch up with the most recent messages of the leader are callled In-Sync replica Leader replica Follower replica 17/07/2025, 12:58 GitHub - anhthii/kafka-notes: Comprehensible material for those who want to learn about Apache Kafka architecture. https://github.com/anhthii/kafka-notes 19/41


Document 1:
Score: 1.6317880153656006
Content: message the leader has When a leader crashes, one of follower replica will be promoted to become the leader A Follower replica that catch up with the most recent messages of the leader are callled In-Sync replica Leader replica Follower replica 17/07/2025, 12:58 GitHub - anhthii/kafka-notes: Comprehensible material for those who want to learn about Apache Kafka architecture. https://github.com/anhthii/kafka-notes 19/41


Document 2:
Score: 1.7076423168182373
Content: don't nec

In [4]:
retriver = vector_db.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retriver

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000286EDC19E80>, search_kwargs={'k': 3})

In [5]:
from langchain_core.prompts import ChatPromptTemplate
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.
Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [6]:
import os
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0,
)


In [7]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

documents_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
documents_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18', 'langchain-google-genai': '4.4.0'}}, p

In [8]:
from langchain_classic.chains import create_retrieval_chain

reg_chain = create_retrieval_chain(retriver, documents_chain)
reg_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000286EDC19E80>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf

In [9]:
query = "How kafka maintain ordering?"
response = reg_chain.invoke({"input": query})
response['answer']

C:\RAG\Code\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


'Kafka ensures message ordering within a partition by using a specified message key. Consumers then read topic messages in the exact order in which they were produced. To maintain position and ordering during consumption, Kafka uses an offset integer to track messages and ensures that each partition is consumed by only one member of a consumer group.'

In [10]:
query = "What is partitioning?"
response = reg_chain.invoke({"input": query})
response['answer']

C:\RAG\Code\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'In Kafka, partitions group data by key. They are composed of files called segments, where the data is stored. An active segment within a partition is the segment currently being written to.'

## RAG Chain Alternative : LCEL

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

custom_prompt = ChatPromptTemplate.from_template(""" Use the following context to answer the question. If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.
Context : {context}
Question: {question}
Answer : 
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

get_question = RunnableLambda(lambda inputs: inputs["question"])

reg_chain_lcel = ({"context": get_question | retriver | format_docs,
                   "question": get_question}
                  | custom_prompt
                  | llm
                  | StrOutputParser()
                  )

reg_chain_lcel

{
  context: RunnableLambda(lambda inputs: inputs['question'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000286EDC19E80>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnableLambda(lambda inputs: inputs['question'])
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template=" Use the following context to answer the question. If you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\nContext : {context}\nQuestion: {question}\nAnswer : \n"), additional_kwargs={})])
| ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18', 'langchain-google-g

In [12]:
response = reg_chain_lcel.invoke({"question": "If the number of consumer in a group is exceeds then what will happen?"})
response

C:\RAG\Code\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the provided context, if the number of consumers in a group exceeds the number of partitions in a topic, "there will be some idle consumer that get no messages at all."'

In [13]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage


contextualized_q_system_promt = """
Given a chat history and the latest user question, which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, just rephrase it if needed and otherwise return it as is.
"""

contextualized_q_prompt = ChatPromptTemplate.from_messages([("system", contextualized_q_system_promt),MessagesPlaceholder("chat_history"),("human", "{input}")])

history_aware_retriver = create_history_aware_retriever(llm=llm,retriever=retriver,prompt=contextualized_q_prompt)

In [14]:
# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.
Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([("system", qa_system_prompt),MessagesPlaceholder("chat_history"),("human", "{input}"),])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create conversational RAG chain
conversational_rag_chain = create_retrieval_chain(history_aware_retriver,question_answer_chain)

In [15]:
chat_history = []
# First question
result1 = conversational_rag_chain.invoke({
"chat_history": chat_history,
"input": "What is Message Broker?"
})
print(f"A: {result1['answer']}")

chat_history.extend([HumanMessage(content="What is Message Broker?"),AIMessage(content=result1['answer'])])
## Follow up question
result2 = conversational_rag_chain.invoke({"chat_history": chat_history,
"input": "What are the partitions?" # Refers to ML from previous question
})
result2['answer']

C:\RAG\Code\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


A: A message broker is an intermediary that brings together two parties that do not necessarily know each other for a mutually beneficial exchange or deal. In Kafka, a broker is a single server that receives messages from producers, assigns offsets to them, and commits the messages to disk storage. These brokers are designed to operate as part of a cluster managed with the help of Apache ZooKeeper.


C:\RAG\Code\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\RAG\Code\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Partitions are components in Kafka that group data by key and are made up of file segments. Kafka assigns messages to a partition using the formula `target_partition = HashCode(key) % number of partitions`. This ensures that all records with the same key are always sent to the same partition in order.'